# ML-02 — Research Question and Provisional Lane
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Divyanshu10045/flyrank-assignments/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)
This notebook frames my capstone research question for the Applied Search Intelligence track. The lane below is *provisional* — I can confirm or change it until the end of Week 4. The sections follow the assignment card in order: lane → the decision/action/cost → a quick checked look at the starter data → the limits of what I can claim → self-check. Everything here is framing and observed counts — no modeling yet.

## 1. My lane (or freestyle) and why
**Lane: Lane 2 — Refresh / Content Opportunity Scoring.**
I want the capstone to answer one practical question: *which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring?* — a ranked action queue, not a model for its own sake. I picked this lane because it ends in a clear human decision: an editor has limited review capacity, and the queue says where that capacity goes first, with a reason code a human can inspect. Lane 1 (signal analysis) finishes as a report rather than a decision; Lane 3 (clustering) is a useful lens but not a decision on its own; Lane 4 (CTR/engagement) is a narrower slice of the same queue-building job. Lane 2 covers that ground while still allowing a stronger capstone label (prior 90 days of features → an outcome observed in the next 30 days) if the numbers below hold up on the full release.
The starter numbers in Section 3 sealed the choice: more than half of the pages are marked declining, and most of them still carry real search demand. A pool that big needs prioritization, not a flat list.

In [1]:
# Setup once: climb to the repo root so the relative data path works from anywhere this kernel starts.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Why the pool is too big to eyeball: declining pages with real demand, vs. a reviewer's real capacity.
down_with_demand = df[(df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)]
review_capacity = 40  # assumed pages/week a reviewer can actually act on
print(f"declining pages with >= 100 impressions/90d: {len(down_with_demand):,}")
print(f"at ~{review_capacity} pages/week, that pool alone needs ~{len(down_with_demand) / review_capacity:.0f} weeks of review")

declining pages with >= 100 impressions/90d: 13,152
at ~40 pages/week, that pool alone needs ~329 weeks of review


## 2. The question: decision, action, cost of a wrong call
- **Decision it improves:** *which page does a content reviewer look at first.* The output is a ranked review queue — a priority order over the inventory, not a prediction to file away.
- **Who acts, and what they do:** a content editor / SEO reviewer opens the queue, reads the reason code on each top-ranked page, and spends that week's review capacity on the top of the list. The per-page action can then be refresh, expand, protect, prune, or monitor.
- **Cost of a wrong call:** a page wrongly ranked high burns an editor's session on something that was never going to move; a high-demand declining page wrongly ranked low stays untouched while its traffic keeps slipping. Because review capacity is a fixed top-K, the cost is asymmetric — missing an impactful page costs more than wasting one — so the method must favor precision at the very top of the queue.
- **Output:** a ranked per-page queue with a suggested action, a human-readable reason code, a confidence label, and a model/analysis card a reviewer could audit.
- **Why data/ML helps at all:** the pool is thousands of pages wide and the signals are many and tangled (demand, position, CTR, freshness, age, competition, intent). A plain if-statement can make a reasonable first cut — that is the baseline we will try to beat — but a learned ranking can combine more evidence at once. On this starter slice the repo's own starter pipeline already reports the learned model beating the rule baseline on precision at the top of the queue (Precision@50 0.240 → 0.740, `outputs/model_report.md`). That is a directional hint, not a promise — the result has to be earned again on the full release with proper validation.

In [2]:
# The declining pool is not one uniform mass: it runs from low-demand to excellent-demand pages,
# which is why a flat “declining” flag is not enough — impact must shape the ranking.
import os
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

tiers = ["no_data", "none", "low", "moderate", "good", "excellent"]
comp = df[df["trend_direction"] == "down"]["impression_tier"].value_counts().reindex(tiers).fillna(0).astype(int)
print(comp.to_string())

impression_tier
no_data         0
none            0
low          5106
moderate     6435
good         4223
excellent     498


## 3. Quick look at the data (2–3 real numbers)
Observed on the 30,000-row starter slice (`data/raw/content_refresh_anonymized.csv`, trailing-90-day window at export). Each number is recomputed by the executed cell below — nothing is carried in from memory:
1. **54.2% of pages are marked `trend_direction = 'down'`** (16,262 of 30,000). Decline is the common case in this slice, not a rare event — a plain “flag declining pages” system would flag most of the inventory and help no one.
2. **13,152 of those declining pages still have real demand** (≥ 100 impressions in the last 90 days, ~43.8% of all pages). This is the triage pool an editor would face — far more than any reviewer can actually look at.
3. **That pool still earns ~80M search impressions per 90 days, with a median of ~1,620 per page.** These pages are losing ground *while still being seen* — the exposure at risk is real, so the cost of not prioritizing is a continuing loss, not zero.
Why this earns the next 7 weeks: ranking ~13k impactful, declining pages for a reviewer who can act on a few dozen a week is a decision-support problem with a named customer and a concrete cost of getting the order wrong. That is exactly Lane 2's deliverable. These are snapshot counts from a teaching slice at one point in time — the warehouse work later re-checks the same question at scale.

In [3]:
# The 2–3 headline numbers for Section 3, computed in one place.
import os
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

n = len(df)
down = df["trend_direction"] == "down"
pool = df[down & (df["impressions_90d"] >= 100)]

print(f"rows: {n:,}")
print(f"1) declining pages: {int(down.sum()):,}  = {down.mean() * 100:.1f}% of pages")
print(f"2) declining with >= 100 impressions/90d: {len(pool):,}  = {len(pool) / n * 100:.1f}% of all pages")
print(f"3) pooled 90d impressions of that pool: {int(pool['impressions_90d'].sum()):,}")
print(f"   median 90d impressions per page in that pool: {pool['impressions_90d'].median():.1f}")

rows: 30,000
1) declining pages: 16,262  = 54.2% of pages
2) declining with >= 100 impressions/90d: 13,152  = 43.8% of all pages
3) pooled 90d impressions of that pool: 79,887,612
   median 90d impressions per page in that pool: 1620.5


## 4. Careful words: what I can and can't claim
What this line of work **can** claim, when the evidence holds up:
- **Observed / measured:** counts from executed output, e.g. "54.2% of pages in this snapshot are marked declining in the trailing window." These describe this pseudonymized slice at one point in time — they are not universal facts about search.
- **Directional:** "declining pages in this slice still hold real demand" — an association seen in the data, never a mechanism.
- **Decision-support:** "given these observed signals, the queue ranks this page #4 for review and suggests 'refresh' with reason code X." A prioritization aid for a human reviewer, not a verdict.
What this work **cannot** claim, no matter how clean the numbers look:
- That refreshing a page *caused* a recovery, or that any page is *guaranteed* to recover — that needs an experiment, and this observational data cannot run one.
- Anything about Google's (or any search engine's) actual algorithm, ranking factors, or AI citation behavior. The data is search/analytics observations — it is not an instrumented test of an algorithm.
- That the starter `trend_direction` bucket is the ground truth on decline — it is a current-window proxy. The capstone's real target (from the Week 3 data contract onward) is a future-window outcome: features from a prior window → an outcome observed in a later window.
- Any identity-level claim: `client_id` / `content_id` are pseudonyms used for grouping and held-out splits only. No client name, URL, query, or private data appears anywhere in this work.
The discipline: every number above comes from a cell I actually ran in this notebook. If a cell errors, I fix it and re-run it — I never write down a plausible-looking value.

In [4]:
# Careful-words guard: re-derive the figures and confirm the slice is safe to work with.
import os
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("rows x cols:", df.shape)
print("distinct clients:", df["client_id"].nunique(), "(pseudonymized ids only)")
risk = [c for c in df.columns if any(k in c.lower() for k in ("url", "query", "name", "domain"))]
print("columns that could expose client data:", risk if risk else "none")

print("\nre-deriving Section 3 figures from this executed cell:")
down = df["trend_direction"] == "down"
pool = df[down & (df["impressions_90d"] >= 100)]
print(f"  declining: {int(down.sum()):,}")
print(f"  declining with >= 100 impressions/90d: {len(pool):,}")
print(f"  pooled 90d impressions: {int(pool['impressions_90d'].sum()):,}")

rows x cols: (30000, 44)
distinct clients: 32 (pseudonymized ids only)
columns that could expose client data: none

re-deriving Section 3 figures from this executed cell:
  declining: 16,262
  declining with >= 100 impressions/90d: 13,152
  pooled 90d impressions: 79,887,612


## Self-check
Before submitting, I confirm each line honestly:
- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit the repo URL on the card. Done.